# Finetune Sunflower-Gemma for Kikuyu EN->KI (LoRA)
Run on Colab with **T4 GPU**: Runtime > Change runtime type > T4.
1. Accept Gemma terms: https://huggingface.co/models/google/gemma-3-4b-it
2. Add Colab secret `HF_TOKEN` (fine-grained, write) — left key icon.
3. Run all. Pilot default 5k rows (~30-45 min on T4). Scale MAX_ROWS to 30000-100000 after pilot.

In [ ]:
!pip install -q transformers peft trl datasets sacrebleu bitsandbytes accelerate

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - enable T4 first')
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('logged in')

In [ ]:
LANG, LANG_NAME = 'kik', 'Kikuyu'
DATASET = 'BlakHasan/kenyan-corpus-kikuyu'  # 1.91M rows
BASE_MODEL = 'google/gemma-3-4b-it'
PUSH_REPO = 'BlakHasan/sunflower-mt-kik'
MAX_ROWS, MIN_SIM, SEED = 5000, 1.0, 42  # pilot; raise to 30000+ after
EPOCHS, LR = 1.0, 2e-4

import pandas as pd
from datasets import load_dataset
SYSTEM = 'You are Sunflower, a helpful assistant made by Sunbird AI who knows many African languages.'
ds = load_dataset(DATASET, split='train', streaming=True)
rows = [r for i, r in enumerate(ds) if i < MAX_ROWS]
df = pd.DataFrame(rows)
s_c = next((c for c in df.columns if c.lower() in ('source_text','src','source','english')), list(df.columns)[0])
t_c = next((c for c in df.columns if c.lower() in ('target_text','tgt','target','kikuyu','kik')), list(df.columns)[1])
pairs = pd.DataFrame({'src': df[s_c].astype(str).str.strip(), 'tgt': df[t_c].astype(str).str.strip()})
pairs = pairs[(pairs.src.str.len()>=3)&(pairs.tgt.str.len()>=3)].drop_duplicates().reset_index(drop=True)
print(f'pairs: {len(pairs)}')
chat = [{'messages': [{'role':'system','content':SYSTEM},{'role':'user','content':f'Translate from English to {LANG_NAME}: '+s},{'role':'assistant','content':t}]} for s,t in zip(pairs.src, pairs.tgt)]
chat = pd.DataFrame(chat)
test = chat.sample(frac=0.02, random_state=SEED)
train = chat.drop(test.index)
print(f'train={len(train)} eval={len(test)}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, load_in_4bit=True, device_map='auto')
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
print('model + LoRA ready')

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
args = SFTConfig(output_dir='./out-kik', num_train_epochs=EPOCHS, learning_rate=LR, per_device_train_batch_size=4, gradient_accumulation_steps=8, logging_steps=50, save_steps=500, eval_strategy='steps', eval_steps=500, bf16=True, optim='paged_adamw_8bit', max_seq_length=1024, push_to_hub=True, hub_model_id=PUSH_REPO)
trainer = SFTTrainer(model=model, train_dataset=Dataset.from_pandas(train[['messages']]), eval_dataset=Dataset.from_pandas(test[['messages']]), args=args)
trainer.train()
trainer.push_to_hub()
print(f'pushed -> https://huggingface.co/{PUSH_REPO}')

In [ ]:
# Quick chrF sanity on 200 held-out pairs (full FLORES kik devtest after)
import sacrebleu
sample = test.sample(min(200, len(test)), random_state=0)
preds = []
for m in sample['messages']:
    src = m[1]['content'].split(': ',1)[1]
    inp = tok.apply_chat_template([m[0], m[1]], tokenize=False, add_generation_prompt=True)
    ids = tok(inp, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=128, do_sample=False)
    preds.append(tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip())
refs = [m[2]['content'] for m in sample['messages']]
print('chrF:', sacrebleu.corpus_chrf(preds, [refs]).score)